# Template

In [7]:
import polars as pl
from black.trans import defaultdict

import src.social_groups.polars_columns as plc
from social_groups.analysis.defs.notebooks.definitions import register_materialization
from social_groups.analysis.polars_transformations.apply_parsing_and_group_decision import (
    apply_parsing_and_group_decision,
)
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
)
from social_groups.reporting.parsing import (
    AnswerComparer,
    AnswerOptions,
    AnswerParser,
)

In [8]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling="wrong"
)

In [9]:
from social_groups.analysis.definitions import defs

frame: pl.DataFrame = defs().load_asset_value("changed_order_mad")

2026-03-20 15:38:46 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/changed_order_mad.parquet using PolarsParquetIOManager...


In [10]:
parsed_data = apply_parsing_and_group_decision(frame, parser, comparer, group_reply)
data = (
    parsed_data.group_by(plc.group_constellation)
    .agg(pl.col(plc.is_correct).mean().alias(plc.accuracy))
    .sort(plc.accuracy, plc.group_constellation, descending=True)
)

register_materialization(
    "changed_order_mad_evaluation_table",
    data,
    "Accuracy per group constellation for different orderings of models using MAD.",
)

data

Skipping Materialization because in interactive mode.


/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagstermill/manager.py:286: BetaWarning: Class `DagstermillExecutionContext` is currently in beta, and may have breaking changes in minor version releases, with behavior changes in patch releases.
  self.context = DagstermillExecutionContext(
/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagster/_core/execution/context_creation_job.py:276: RuntimeWarning: coroutine 'BaseEventLoop.shutdown_asyncgens' was never awaited
  pass


group_constellation,accuracy
str,f64
"""MHH""",0.73
"""LHH""",0.71
"""HLH""",0.71
"""MLH""",0.7
"""HHH""",0.7
…,…
"""LML""",0.47
"""MLL""",0.46
"""ML""",0.43


In [11]:
groups: dict[str, list[str]] = defaultdict(list)

for group in data["group_constellation"]:
    split = "".join(sorted(group))
    groups[split].append(group)

for group, parts in groups.items():
    group_data = data.filter(pl.col(plc.group_constellation).is_in(parts)).head()

    group_data.sort("accuracy")

    print(
        " -> ".join(
            f"{group_data['group_constellation'][i]} ({group_data['accuracy'][i]:.2f})"
            for i in range(len(group_data))
        )
    )

MHH (0.73) -> HMH (0.69) -> HHM (0.61)
LHH (0.71) -> HLH (0.71) -> HHL (0.64)
MLH (0.70) -> LMH (0.67) -> HLM (0.65) -> LHM (0.62) -> HML (0.62)
HHH (0.70)
MMH (0.67) -> MHM (0.65) -> HMM (0.63)
HH (0.67)
MH (0.66) -> HM (0.66)
LLH (0.66) -> LHL (0.53) -> HLL (0.51)
LH (0.64) -> HL (0.51)
MML (0.58) -> LMM (0.58) -> MLM (0.57)
MMM (0.57)
LLM (0.57) -> LML (0.47) -> MLL (0.46)
MM (0.55)
LM (0.54) -> ML (0.43)
LLL (0.34)
LL (0.34)


-> Very significant: "Smarter model last" -> "Better results

- MHH is even better than HHH

### But Why?

Wouldn't it be more logical to use a better model sooner to bias the debate towards correctness?


In [105]:
relevant_data_for_group = parsed_data.sort(plc.group_constellation)

# .filter(
#     pl.col(plc.group_constellation).is_in({"MHH", "HMH", "HHM"})
# )

unequal_questions = (
    relevant_data_for_group.pivot(
        index="question_id", values=plc.is_correct, on=plc.group_constellation
    )
    .sort("question_id")
    .with_columns(
        all_equal=pl.mean_horizontal(pl.col("MHH"), pl.col("HMH"), pl.col("HHM"))
        == pl.col("MHH")
    )
    .filter(pl.col("all_equal").not_())
    .drop("all_equal")
    .join(relevant_data_for_group["question_id", "phoenix_span_url"], on="question_id")
    .group_by("question_id")
    .agg(
        pl.exclude("phoenix_span_url").unique().item(),
        pl.col("phoenix_span_url").implode(),
    )
    .sort("question_id")
)
unequal_questions

question_id,HH,HHH,HHL,HHM,HL,HLH,HLL,HLM,HM,HMH,HML,HMM,LH,LHH,LHL,LHM,LL,LLH,LLL,LLM,LM,LMH,LML,LMM,MH,MHH,MHL,MHM,ML,MLH,MLL,MLM,MM,MMH,MML,MMM,phoenix_span_url
i64,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,list[str]
5,true,true,true,true,true,true,false,true,true,false,true,true,true,false,true,false,false,true,false,true,true,false,false,true,false,true,false,true,true,true,true,true,false,true,true,true,"[""http://localhost:6006/projects/UHJvamVjdDoyNTA=/spans/6b6a40b18dfb60ead5308eabe26cd71d?selectedNoteSpanId=U3BhbjozODA2NTg="", ""http://localhost:6006/projects/UHJvamVjdDo0NzI=/spans/6ba0987117f2d431e19e87a325d08e3e?selectedNoteSpanId=U3Bhbjo5MjA0NjA="", … ""http://localhost:6006/projects/UHJvamVjdDo0NzI=/spans/d428f24d29b050b6aa39758f2a3e46e7?selectedNoteSpanId=U3Bhbjo3NzkzMTg=""]"
7,true,true,true,false,true,true,false,false,true,true,true,false,false,true,true,false,false,true,false,false,false,true,true,true,true,false,true,true,true,true,false,true,false,true,false,true,"[""http://localhost:6006/projects/UHJvamVjdDoyNTA=/spans/24ecbbfc7366bfd098f4b9ef2cd2dfde?selectedNoteSpanId=U3BhbjozODMxMDk="", ""http://localhost:6006/projects/UHJvamVjdDo0NzI=/spans/6919dfe39c7855245ae535daf5c43c21?selectedNoteSpanId=U3Bhbjo5MzMwOTA="", … ""http://localhost:6006/projects/UHJvamVjdDo0NzI=/spans/ac426f8ae7c36fa2eeb27dc396928a18?selectedNoteSpanId=U3Bhbjo3ODU5NTY=""]"
21,false,true,false,false,false,true,false,false,false,true,false,false,false,true,false,false,false,true,false,false,false,true,false,false,false,true,false,false,false,false,false,false,false,false,false,false,"[""http://localhost:6006/projects/UHJvamVjdDoyNTA=/spans/7349891ff476126af9376cce5fa2a672?selectedNoteSpanId=U3BhbjozNzgxMjE="", ""http://localhost:6006/projects/UHJvamVjdDo0NzI=/spans/19940f579269076a62c90b1e94891bc0?selectedNoteSpanId=U3Bhbjo5MDU0NDk="", … ""http://localhost:6006/projects/UHJvamVjdDo0NzI=/spans/bfbcd639c931742dbb4304c498f95932?selectedNoteSpanId=U3Bhbjo3NjQxNDA=""]"
24,true,true,false,false,false,true,false,false,false,true,false,true,true,true,true,false,false,true,false,false,false,true,false,false,true,false,false,false,false,true,false,false,false,true,false,false,"[""http://localhost:6006/projects/UHJvamVjdDoyNTA=/spans/8c3ad5a66efcf8181611c7067a76cc54?selectedNoteSpanId=U3BhbjozNzc5MjU="", ""http://localhost:6006/projects/UHJvamVjdDo0NzI=/spans/e454d365a8e325a60944034f10c91e27?selectedNoteSpanId=U3Bhbjo5MDU3MjM="", … ""http://localhost:6006/projects/UHJvamVjdDo0NzI=/spans/77b6078f8f9edef75ef382af561515ae?selectedNoteSpanId=U3Bhbjo3NzI2NzQ=""]"
26,true,true,true,true,true,true,true,true,true,false,true,false,true,true,true,true,true,true,false,true,true,true,false,true,true,true,true,true,true,true,true,false,true,true,true,false,"[""http://localhost:6006/projects/UHJvamVjdDoyNTA=/spans/f38424558ce8f4ab862437c07613dc3c?selectedNoteSpanId=U3BhbjozNzY4NDA="", ""http://localhost:6006/projects/UHJvamVjdDo0NzI=/spans/0ad46af75864689e259bc6bfe6edf4c4?selectedNoteSpanId=U3Bhbjo5MDcyMjE="", … ""http://localhost:6006/projects/UHJvamVjdDo0NzI=/spans/a9db976441b626583831a0c910fab593?selectedNoteSpanId=U3Bhbjo3NjY4NDk=""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
112,false,true,false,false,false,true,false,false,false,false,false,false,true,true,false,false,false,true,false,false,false,true,false,false,true,true,true,true,false,true,false,false,false,true,false,false,"[""http://localhost:6006/projects/UHJvamVjdDoyNTA=/spans/34e624a3b61266a3c85bdb10e0690e88?selectedNoteSpanId=U3BhbjozODk4Mjk="", ""http://localhost:6006/projects/UHJvamVjdDo0NzI=/spans/5f9d49043d1e0a91ea2a9a56ea70f978?selectedNoteSpanId=U3Bhbjo5NjUzMjA="", … ""http://localhost:6006/projects/UHJvamVjdDo0NzI=/spans/4411079c384f8da9e732d96af5bfaeb5?selectedNoteSpanId=U3Bhbjo4MjExNzk=""]"
114,true,true,true,false,fal

In [106]:
letter_ordering = {"H": 2, "M": 1, "L": 0}
ordering = sorted(
    list(
        unequal_questions.select(pl.exclude("question_id", "phoenix_span_url")).columns
    ),
    key=lambda g: (tuple(letter_ordering[x] for x in g)),
)
ordered_unequal_questions = unequal_questions["question_id", *ordering]
ordered_unequal_questions

question_id,LL,LLL,LLM,LLH,LM,LML,LMM,LMH,LH,LHL,LHM,LHH,ML,MLL,MLM,MLH,MM,MML,MMM,MMH,MH,MHL,MHM,MHH,HL,HLL,HLM,HLH,HM,HML,HMM,HMH,HH,HHL,HHM,HHH
i64,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
5,false,false,true,true,true,false,true,false,true,true,false,false,true,true,true,true,false,true,true,true,false,false,true,true,true,false,true,true,true,true,true,false,true,true,true,true
7,false,false,false,true,false,true,true,true,false,true,false,true,true,false,true,true,false,false,true,true,true,true,true,false,true,false,false,true,true,true,false,true,true,true,false,true
21,false,false,false,true,false,false,false,true,false,false,false,true,false,false,false,false,false,false,false,false,false,false,false,true,false,false,false,true,false,false,false,true,false,false,false,true
24,false,false,false,true,false,false,false,true,true,true,false,true,false,false,false,true,false,false,false,true,true,false,false,false,false,false,false,true,false,false,true,true,true,false,false,true
26,true,false,true,true,true,false,true,true,true,true,true,true,true,true,false,true,true,true,false,true,true,true,true,true,true,true,true,true,true,true,false,false,true,true,true,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
112,false,false,false,true,false,false,false,true,true,false,false,true,false,false,false,true,false,false,false,true,true,true,true,true,false,false,false,true,false,false,false,false,false,false,false,true
114,true,false,true,true,false,false,false,false,true,false,false,true,false,true,false,false,false,false,false,false,true,false,false,true,false,false,true,true,true,false,false,false,true,true,false,true
116,false,false,false,false,false,false,false,true,false,false,true,false,false,false,false,true,true,false,false,false,false,false,true,false,false,false,true,false,false,true,true,true,false,false,false,false


In [107]:
cols = ordered_unequal_questions.select(pl.exclude("question_id")).columns
(
    pl.DataFrame({"condition": cols})
    .join(pl.DataFrame(cols), how="cross")
    .select(cond_col=pl.col("condition"), target_col=pl.col("column_0"))
    .map_rows(
        lambda row: (
            row[0],
            row[1],
            ordered_unequal_questions.filter(pl.col(row[0]) == True)
            .select(pl.col(row[1]).mean())
            .item(),
        )
    )
    .rename({"column_0": "condition", "column_1": "target", "column_2": "prob"})
    .pivot(index="condition", on="target", values="prob", aggregate_function="item")
    .with_columns(
        sort_key=pl.col("condition").map_elements(
            lambda g: tuple(letter_ordering[x] for x in g),
            return_dtype=pl.List(pl.Int64),
        )
    )
    .sort("sort_key")
    .drop("sort_key")
)

condition,LL,LLL,LLM,LLH,LM,LML,LMM,LMH,LH,LHL,LHM,LHH,ML,MLL,MLM,MLH,MM,MML,MMM,MMH,MH,MHL,MHM,MHH,HL,HLL,HLM,HLH,HM,HML,HMM,HMH,HH,HHL,HHM,HHH
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""LL""",1.0,0.142857,0.714286,0.571429,0.714286,0.142857,0.571429,0.571429,0.857143,0.428571,0.428571,0.571429,0.285714,0.714286,0.428571,0.571429,0.714286,0.428571,0.285714,0.428571,0.714286,0.571429,0.428571,0.857143,0.714286,0.714286,0.714286,0.857143,0.714286,0.571429,0.285714,0.285714,0.714286,0.571429,0.428571,0.571429
"""LLL""",0.25,1.0,0.75,0.5,0.5,0.0,0.0,0.5,0.5,0.5,0.5,0.5,0.25,0.25,0.5,0.75,0.5,0.5,0.5,0.5,0.75,0.5,0.25,0.75,0.5,0.5,0.75,0.75,0.75,0.5,0.25,0.5,0.75,0.25,0.5,0.75
"""LLM""",0.416667,0.25,1.0,0.75,0.833333,0.25,0.583333,0.75,0.666667,0.583333,0.666667,0.75,0.5,0.5,0.75,0.833333,0.666667,0.75,0.666667,0.75,0.833333,0.5,0.666667,0.833333,0.666667,0.5,1.0,1.0,1.0,0.833333,0.666667,0.416667,0.916667,0.75,0.583333,0.75
"""LLH""",0.222222,0.111111,0.5,1.0,0.444444,0.277778,0.388889,0.833333,0.555556,0.5,0.611111,0.777778,0.277778,0.277778,0.5,0.777778,0.388889,0.388889,0.444444,0.777778,0.777778,0.444444,0.611111,0.722222,0.444444,0.277778,0.555556,0.944444,0.666667,0.611111,0.555556,0.611111,0.833333,0.611111,0.388889,0.888889
"""LM""",0.416667,0.166667,0.833333,0.666667,1.0,0.25,0.666667,0.666667,0.583333,0.583333,0.583333,0.666667,0.5,0.5,0.833333,0.75,0.75,0.75,0.583333,0.75,0.666667,0.583333,0.75,0.833333,0.75,0.416667,0.833333,0.916667,0.833333,0.75,0.666667,0.416667,0.75,0.666667,0.583333,0.666667
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HMH""",0.125,0.125,0.3125,0.6875,0.3125,0.1875,0.3125,0.75,0.375,0.375,0.5,0.75,0.1875,0.25,0.4375,0.8125,0.3125,0.25,0.375,0.75,0.6875,0.375,0.625,0.5,0.3125,0.1875,0.4375,0.75,0.5625,0.5,0.625,1.0,0.6875,0.5,0.1875,0.75
"""HH""",0.294118,0.176471,0.647059,0.882353,0.529412,0.294118,0.470588,0.823529,0.647059,0.588235,0.647059,0.764706,0.352941,0.411765,0.588235,0.882353,0.411765,0.470588,0.529412,0.823529,0.882353,0.470588,0.647059,0.764706,0.588235,0.352941,0.705882,0.941176,0.823529,0.705882,0.588235,0.647059,1.0,0.764706,0.352941,0.823529
"""HHL""",0.307692,0.076923,0.692308,0.846154,0.615385,0.384615,0.615385,0.769231,0.692308,0.615385,0.615385,0.769231,0.461538,0.538462,0.615385,0.846154,0.384615,0.538462,0.538462,0.846154,0.846154,0.538462,0.769231,0.846154,0.692308,0.307692,0.769231,0.923077,0.923077,0.769231,0.615385,0.615385,1.0,1.0,0.307692,0.769231


In [80]:
def build_conditional_prob_table(df: pl.DataFrame) -> pl.DataFrame:
    cols = df.columns

    probs = (
        pl.DataFrame({"condition": cols})
        .join(pl.DataFrame(cols), how="cross")
        .with_columns(cond_col=pl.col("condition"), target_col=pl.col("column"))
        .map_rows(
            lambda row: (
                row[0],  # condition name
                row[1],  # target name
                df.filter(pl.col(row[0]) == True).select(pl.col(row[1]).mean()).item(),
            )
        )
        .rename({"column_1": "condition", "column_2": "target", "column_3": "prob"})
        .pivot(
            index="condition",
            columns="target",
            values="prob",
            aggregate_function="item",
        )
        .with_columns(
            pl.col(cols).round(4)  # optional: nicer display
        )
    )

    # Make condition column the index-like first column
    probs = probs.select(["condition", *cols])

    return probs

In [76]:
print(
    "Probability of Column being correct, after Row is correct. Given that not all answers are the same."
)

ordered_unequal_questions.select(pl.exclude("question_id")).group_by(pl.all()).agg(
    pl.all().implode()
)

Probability of Column being correct, after Row is correct. Given that not all answers are the same.


MHH,HMH,HHM
bool,bool,bool
true,false,false
true,false,true
false,true,false
false,true,true
true,true,false
